# AI Content Claim Check

Checks an AI-written story draft against the source finding it was written from. Every claim in the draft gets a verdict:
**Supported**, **Unsupported** or **Needs Human Verification**. Nothing is published or sent; a human approves every draft.

**How to use this notebook:** run the cells from top to bottom. The notebook first runs the free checks, then **one** case
with the model, and only then all nine cases.

**Before you start:** you need a free Gemini API key (https://aistudio.google.com/apikey).
In Colab, click the key icon in the left sidebar, add a secret named `GEMINI_API_KEY`, paste the key and switch on
"Notebook access". If the repository is private, add a second secret `GITHUB_TOKEN` (read access to this repository only).

### Step 1. Get the code and install the packages
Copies the code from GitHub into Colab and installs the Python packages. This takes about a minute.

In [ ]:
import os

# Address of the repository with the code, and the folder to put it in
REPO_URL = "https://github.com/Yul04ek/ai-content-claim-check.git"
FOLDER = "/content/claim-check"

if not os.path.exists(FOLDER):
    # A private repository needs a token. A public one works without it.
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        clone_url = REPO_URL.replace("https://", f"https://{token}@")
    except Exception:
        clone_url = REPO_URL
    !git clone -q {clone_url} {FOLDER}
    !git -C {FOLDER} remote set-url origin {REPO_URL}   # do not keep the token in the repo settings
%cd {FOLDER}

# Install the Python packages listed in the requirements file
!pip install -q -r requirements.txt

### Step 2. Connect the model
Reads your key from Colab Secrets, sets the options below and sends one short test message. The test should print `ready`.

| Setting | Meaning |
|---|---|
| `MODEL_NAME` | Gemini model used as the checker. Keep the same model for all nine cases. If it is overloaded (error 503), try another one from AI Studio. |
| `BASE_URL` | Gemini through the OpenAI-compatible API. |

In [ ]:
import os
from google.colab import userdata

# Read the key from Colab Secrets
try:
    os.environ["API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    raise RuntimeError(
        "Secret GEMINI_API_KEY not found. Add it with the key icon in the left sidebar, "
        "switch on 'Notebook access', and run this cell again."
    )

os.environ["BASE_URL"] = "https://generativelanguage.googleapis.com/v1beta/openai/"  # Gemini through the OpenAI-compatible API
os.environ["MODEL_NAME"] = "gemini-3.5-flash-lite"   # the model that worked in the research agent; change if needed

# Send one short message to check that the key and the model work (30 second limit)
from openai import OpenAI

client = OpenAI(base_url=os.environ["BASE_URL"], api_key=os.environ["API_KEY"]).with_options(timeout=30, max_retries=0)
reply = client.chat.completions.create(
    model=os.environ["MODEL_NAME"],
    messages=[{"role": "user", "content": "Reply with the single word: ready"}],
)
print(reply.choices[0].message.content)

### Step 3. Free checks (no model call)
The offline tests should end with `20 tests passed`. The dry run uses only the checks done in plain Python, so its result is
**not** a verdict; it shows that the cases and the data are read correctly.

In [ ]:
!python tests/test_claim_check.py
!python -m claim_check --dry-run --out results/dryrun.csv

### Step 4. First run with the model: one case
Case T6 is a draft with a deliberately invented number (`30%`) and a fix described as finished. Expected: those claims are Unsupported.
The output ends with the tokens used. If you see a 429 error, wait a minute and run the cell again.

In [ ]:
!python -m claim_check --only "T6 " --out results/results_T6.csv

### Step 5. All nine cases
Each case prints what the checker found, and below it what was expected (the result of the manual review), so you can compare them.
Keep the saved `results/results.csv` of this run for the report.

In [ ]:
!python -m claim_check --out results/results.csv

### Step 6. Look at the results
Shows every row that is not plain "Supported": verdicts, flags and placeholders.

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

results = pd.read_csv("results/results.csv")
print(results.groupby(["case", "verdict"], dropna=False).size().unstack(fill_value=0))
results[results["verdict"] != "Supported"][["case", "kind", "text", "verdict", "finding", "reason"]]

### Step 7. Save the results
Everything in Colab is deleted when the session ends. This cell downloads the results as a zip file.

In [ ]:
!zip -qr /content/claim_check_results.zip results
from google.colab import files
files.download("/content/claim_check_results.zip")